In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [2]:
from src.transforms import trainval_transforms, revert_normalization, revert_standardization
from src.dataset import ImageDataset
import torch

annot_path = Path("../data/preprocessed/trainval/annotations.csv")
img_dir = Path("../data/preprocessed/trainval/images")

# test dataset without transforms
dataset = ImageDataset(annot_path, img_dir, transform=trainval_transforms)

In [3]:
from src.model import Model
from torch.utils.data import DataLoader

trainval_dl = DataLoader(dataset, 8, True)
X_batch, y_batch = next(iter(trainval_dl))
model = Model()

model.eval()
with torch.no_grad():
    preds_batch = model(X_batch)
    
preds_batch.shape, preds_batch

(torch.Size([8, 7, 7, 25]),
 tensor([[[[ 1.5677e-02, -1.2690e-03,  3.0371e-02,  ..., -6.8825e-04,
             2.9382e-02, -1.7700e-02],
           [-1.2491e-02,  2.1401e-02,  6.9168e-03,  ..., -8.1319e-03,
             2.3105e-03,  9.9350e-03],
           [ 9.6536e-03,  2.1406e-03,  9.4094e-03,  ...,  3.0399e-03,
             1.2214e-02, -2.4902e-03],
           ...,
           [-1.2747e-02,  1.6191e-02,  1.5072e-02,  ...,  5.7673e-03,
            -1.1016e-02, -1.1433e-02],
           [-3.9288e-04, -1.0530e-02, -6.1516e-03,  ...,  4.6579e-04,
            -1.4983e-02,  6.5279e-03],
           [ 8.0562e-03, -1.2482e-02,  3.7628e-03,  ..., -1.0319e-02,
            -8.5567e-03,  3.1209e-03]],
 
          [[-1.0961e-03,  6.2843e-03,  3.5371e-03,  ..., -8.9538e-03,
             1.9763e-03,  3.1192e-03],
           [-1.0978e-02,  9.7821e-03,  7.4658e-03,  ...,  1.0873e-02,
             3.6285e-03,  9.7871e-03],
           [ 3.8545e-04, -4.1540e-03,  8.2304e-03,  ..., -6.7619e-03,
           

In [5]:
from src.postprocessing import decode_preds, filter_and_group_preds, sort_class_preds_by_confidence, non_maximum_suppression

final_preds_batch = []
batch_size = preds_batch.shape[0]

for i in range(batch_size):
    preds = preds_batch[i]

    decoded_preds = decode_preds(preds)
    filtered_grouped_preds = filter_and_group_preds(decoded_preds)

    print(filtered_grouped_preds)
    
    sort_class_preds_by_confidence(filtered_grouped_preds)
    suppressed_preds = non_maximum_suppression(filtered_grouped_preds)

    final_preds_batch.append(suppressed_preds)

final_preds_batch

{'cat': [(tensor(0.0002), tensor(32.4796), tensor(-0.2415), tensor(30.6581), tensor(0.2761)), (tensor(0.0002), tensor(96.5262), tensor(158.6565), tensor(94.7289), tensor(160.6997)), (tensor(0.0002), tensor(160.0682), tensor(193.3505), tensor(160.2649), tensor(189.9808))], 'diningtable': [(tensor(0.0002), tensor(159.9242), tensor(1.6882), tensor(160.0285), tensor(-1.6680))], 'car': [(tensor(6.9147e-05), tensor(192.4168), tensor(1.0133), tensor(190.1054), tensor(-0.9034)), (tensor(0.0004), tensor(97.8046), tensor(64.2258), tensor(95.1248), tensor(64.6341))], 'dog': [(tensor(3.4062e-05), tensor(1.1828), tensor(32.1325), tensor(-0.8228), tensor(32.5752)), (tensor(0.0003), tensor(96.5398), tensor(192.4324), tensor(95.4489), tensor(192.4357))], 'horse': [(tensor(0.0002), tensor(30.8835), tensor(30.8913), tensor(33.3190), tensor(31.7041))], 'cow': [(tensor(0.0001), tensor(30.9587), tensor(62.7881), tensor(32.5165), tensor(64.7858))], 'pottedplant': [(tensor(2.5547e-05), tensor(127.1743), tens

[{'cat': [(tensor(0.0002),
    tensor(32.4796),
    tensor(-0.2415),
    tensor(30.6581),
    tensor(0.2761)),
   (tensor(0.0002),
    tensor(160.0682),
    tensor(193.3505),
    tensor(160.2649),
    tensor(189.9808)),
   (tensor(0.0002),
    tensor(96.5262),
    tensor(158.6565),
    tensor(94.7289),
    tensor(160.6997))],
  'diningtable': [(tensor(0.0002),
    tensor(159.9242),
    tensor(1.6882),
    tensor(160.0285),
    tensor(-1.6680))],
  'car': [(tensor(0.0004),
    tensor(97.8046),
    tensor(64.2258),
    tensor(95.1248),
    tensor(64.6341)),
   (tensor(6.9147e-05),
    tensor(192.4168),
    tensor(1.0133),
    tensor(190.1054),
    tensor(-0.9034))],
  'dog': [(tensor(0.0003),
    tensor(96.5398),
    tensor(192.4324),
    tensor(95.4489),
    tensor(192.4357)),
   (tensor(3.4062e-05),
    tensor(1.1828),
    tensor(32.1325),
    tensor(-0.8228),
    tensor(32.5752))],
  'horse': [(tensor(0.0002),
    tensor(30.8835),
    tensor(30.8913),
    tensor(33.3190),
    tensor(3